In [0]:
# Exploring CDC Dataset.

display(dbutils.fs.ls("/Volumes/databricks_simulated_retail_customer_data/v02/customer_changes_daily/"))

path,name,size,modificationTime
dbfs:/Volumes/databricks_simulated_retail_customer_data/v02/customer_changes_daily/customer_changes_2025-11-01.json,customer_changes_2025-11-01.json,135364,1764103610000
dbfs:/Volumes/databricks_simulated_retail_customer_data/v02/customer_changes_daily/customer_changes_2025-11-02.json,customer_changes_2025-11-02.json,146675,1764103610000
dbfs:/Volumes/databricks_simulated_retail_customer_data/v02/customer_changes_daily/customer_changes_2025-11-03.json,customer_changes_2025-11-03.json,136489,1764103610000
dbfs:/Volumes/databricks_simulated_retail_customer_data/v02/customer_changes_daily/customer_changes_2025-11-04.json,customer_changes_2025-11-04.json,148680,1764103611000
dbfs:/Volumes/databricks_simulated_retail_customer_data/v02/customer_changes_daily/customer_changes_2025-11-05.json,customer_changes_2025-11-05.json,151657,1764103611000
dbfs:/Volumes/databricks_simulated_retail_customer_data/v02/customer_changes_daily/customer_changes_2025-11-06.json,customer_changes_2025-11-06.json,124121,1764103611000
dbfs:/Volumes/databricks_simulated_retail_customer_data/v02/customer_changes_daily/customer_changes_2025-11-07.json,customer_changes_2025-11-07.json,142882,1764103612000


In [0]:
# reading CDC Files.

cdc_customers = spark.read.option(
    "multiline",
    "true"
).json("/Volumes/databricks_simulated_retail_customer_data/v02/customer_changes_daily/")

display(cdc_customers)

city,customer_id,email,first_name,last_name,loyalty_tier,operation,signup_date,source_subsidiary,timestamp
Denver,CUST_00019,alicia.torres00019@example.com,Alicia,Torres,bronze,new,2025-11-05,lumina_sports,2025-11-05T15:43:19Z
Austin,CUST_00029,olivia.ramos00029+upd05@example.com,Olivia,Ramos,silver,updated,2025-11-02,bright_home,2025-11-05T13:07:11Z
Denver,CUST_00044,evelyn.foster00044+upd05@example.com,Evelyn,Foster,silver,updated,2025-11-01,lumina_sports,2025-11-05T14:04:37Z
Seattle,CUST_00046,james.wright00046@example.com,James,Wright,bronze,new,2025-11-05,lumina_sports,2025-11-05T12:36:34Z
Berlin,CUST_00053,liam.jenkins00053@example.com,Liam,Jenkins,bronze,new,2025-11-05,bright_home,2025-11-05T13:55:18Z
Austin,CUST_00066,amelia.wright00066@example.com,Amelia,Wright,bronze,new,2025-11-05,northstar_outfitters,2025-11-05T15:35:26Z
Denver,CUST_00093,priya.wright00093@example.com,Priya,Wright,silver,new,2025-11-05,northstar_outfitters,2025-11-05T15:21:40Z
Austin,CUST_00110,amelia.ramos00110+upd05@example.com,Amelia,Ramos,silver,updated,2025-11-01,bright_home,2025-11-05T08:30:26Z
London,CUST_00112,emma.patel00112@example.com,Emma,Patel,silver,updated,2025-11-01,northstar_outfitters,2025-11-05T17:03:55Z
Berlin,CUST_00121,priya.carter00121@example.com,Priya,Carter,silver,new,2025-11-05,lumina_sports,2025-11-05T15:26:31Z


In [0]:
cdc_customers.printSchema()

root
 |-- city: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- loyalty_tier: string (nullable = true)
 |-- operation: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- source_subsidiary: string (nullable = true)
 |-- timestamp: string (nullable = true)



In [0]:
# creating SCD2 table.

spark.sql(""" create table if not exists retail_project.customer_scd2 
(
    customer_id LONG,
    customer_name STRING,
    city STRING,
    loyalty_segment STRING,

    start_date TIMESTAMP,
    end_date TIMESTAMP,

    is_current BOOLEAN
)  using delta """)

DataFrame[]

In [0]:
from pyspark.sql import Row

new_changes = [

    Row(
        customer_id=101,
        customer_name="Piyush",
        city="Bangalore",
        loyalty_segment="Gold"
    ),

    Row(
        customer_id=102,
        customer_name="sudheer",
        city="Noida",
        loyalty_segment="Silver"
    )

]

new_changes_df = spark.createDataFrame(new_changes)

display(new_changes_df)

customer_id,customer_name,city,loyalty_segment
101,Piyush,Bangalore,Gold
102,sudheer,Noida,Silver


In [0]:
from pyspark.sql.functions import current_timestamp

spark.sql("""

update retail_project.customer_scd2

set
    end_date = current_timestamp(),
    is_current = false

where customer_id IN (
    select customer_id
    from retail_project.customer_scd2
)

and is_current = true

""")

DataFrame[num_affected_rows: bigint]

In [0]:
from pyspark.sql.functions import current_timestamp, lit

new_scd_records = new_changes_df.withColumn(
    "start_date",
    current_timestamp()
).withColumn(
    "end_date",
    lit(None).cast("timestamp")
).withColumn(
    "is_current",
    lit(True)
)

new_scd_records.write.format("delta") \
    .mode("append") \
    .saveAsTable("retail_project.customer_scd2")

In [0]:
display(spark.table("retail_project.customer_scd2"))

customer_id,customer_name,city,loyalty_segment,start_date,end_date,is_current
101,Piyush,Bangalore,Gold,2026-05-27T05:44:06.203Z,null,true
102,sudheer,Noida,Silver,2026-05-27T05:44:06.203Z,null,true
